# SSH auth log analysis (Colab)

Run cells **top to bottom**. Upload your course `auth_events.json` in Part 1.

## Part 1 — Upload the JSON file

In [ ]:
from google.colab import files

uploaded = files.upload()  # pick auth_events.json
DATA_FILE = "auth_events.json"
assert DATA_FILE in uploaded, f"Upload a file named {DATA_FILE}"

## Part 2 — Load JSON and print report header

In [ ]:
import json

with open(DATA_FILE, encoding="utf-8") as f:
    report = json.load(f)

server = report["server"]
window = report["window"]
collector = report["collector"]
policy = report["policy"]
last_review = report["last_review"]
events = report["events"]
max_failures = policy["max_failures_per_ip"]
watched_accounts = policy["watched_accounts"]

print(f"Server: {server}")
print(
    f"Observation window: {window['start']} to {window['end']} "
    f"({window['length_hours']} hours)"
)
print(f"Collector version: {collector['version']}")
print(f"last_review: {last_review!r}")
assert last_review is None, "last_review should be Python None, not the string 'null'"
print("Confirmed: last_review is None.")
print(f"Number of events: {len(events)}")

## Part 3 — One pass: build four dictionaries

In [ ]:
failures_by_ip = {}
successes_by_ip = {}
failures_by_user = {}
failures_by_hour = {}

for event in events:
    ip = event["ip"]
    user = event["user"]
    hour = int(event["time"][11:13])

    if event["result"] == "fail":
        failures_by_ip[ip] = failures_by_ip.get(ip, 0) + 1
        failures_by_user[user] = failures_by_user.get(user, 0) + 1
        failures_by_hour[hour] = failures_by_hour.get(hour, 0) + 1
    elif event["result"] == "success":
        successes_by_ip[ip] = successes_by_ip.get(ip, 0) + 1

all_ips = sorted({e["ip"] for e in events})
print("failures_by_ip:", failures_by_ip)
print("successes_by_ip:", successes_by_ip)

## Part 4 — One row per address (policy check)

In [ ]:
print("Per-address summary:")
print(f"{'IP':<18} {'Fails':>6} {'OK':>6}  Assessment")
print("-" * 55)

for ip in all_ips:
    fails = failures_by_ip.get(ip, 0)
    oks = successes_by_ip.get(ip, 0)
    if fails > max_failures:
        note = f"OVER limit ({max_failures})"
    elif fails == max_failures:
        note = f"AT limit ({max_failures})"
    else:
        note = f"within limit ({max_failures})"
    print(f"{ip:<18} {fails:>6} {oks:>6}  {note}")

## Part 5 — Serious case (over limit + success)

In [ ]:
breached = [
    ip for ip in all_ips
    if failures_by_ip.get(ip, 0) > max_failures and successes_by_ip.get(ip, 0) > 0
]

for ip in breached:
    first_ok = next(e for e in events if e["ip"] == ip and e["result"] == "success")
    fails_before = sum(
        1 for e in events
        if e["ip"] == ip and e["result"] == "fail" and e["time"] < first_ok["time"]
    )
    print("*** WARNING: SERIOUS BREACH ***")
    print(f"Address: {ip}")
    print(f"Account reached: {first_ok['user']}")
    print(f"Failures before success: {fails_before}")
    print(
        "Worse than many failures with no login: the attacker got in and can use the account."
    )

if not breached:
    print("No IP both broke the limit and logged in successfully.")

## Part 6 — Watched accounts, peak hour, attack verdict

In [ ]:
print("Failed attempts on watched accounts:")
for name in watched_accounts:
    print(f"  {name}: {failures_by_user.get(name, 0)}")

peak_hour = max(range(24), key=lambda h: failures_by_hour.get(h, 0))
peak_n = failures_by_hour.get(peak_hour, 0)
print(f"\nBusiest hour: {peak_hour:02d}:00 ({peak_n} failures)")

over_limit = {ip for ip in all_ips if failures_by_ip.get(ip, 0) > max_failures}
attacked = len(over_limit) > 0
succeeded = len(breached) > 0
print(f"\nServer attacked? {'YES' if attacked else 'NO'}")
print(f"Attack succeeded? {'YES' if succeeded else 'NO'}")

## Part 7 — Figure (save then show)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

ips = sorted(all_ips, key=lambda ip: failures_by_ip.get(ip, 0))
counts = [failures_by_ip.get(ip, 0) for ip in ips]
colors = ["crimson" if c > max_failures else "steelblue" for c in counts]
hours = list(range(24))
per_hour = [failures_by_hour.get(h, 0) for h in hours]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 9))

ax1.barh(ips, counts, color=colors)
ax1.axvline(max_failures, color="black", linestyle="--", linewidth=1.5)
ax1.set_xlabel("Failed login attempts")
ax1.set_ylabel("Source IP address")
ax1.set_title("Failed attempts per source address")
ax1.grid(True, axis="x", alpha=0.3)
ax1.legend(
    handles=[
        Patch(facecolor="steelblue", label="Within policy limit"),
        Patch(facecolor="crimson", label="Over policy limit"),
        Line2D([0], [0], color="black", linestyle="--", label=f"Limit ({max_failures})"),
    ],
    loc="lower right",
)

ax2.bar(hours, per_hour, color="steelblue", label="Failed attempts")
ax2.set_xlabel("Hour of day")
ax2.set_ylabel("Failed login attempts")
ax2.set_title("Failed attempts by hour of day")
ax2.set_xticks(hours)
ax2.grid(True, axis="y", alpha=0.3)
ax2.legend(loc="upper right")

fig.tight_layout()
fig.savefig("auth_analysis.png", dpi=120)
plt.show()